# Cube+Vector 协同流水

本节建立融合算子的三块基石：混合 Kernel 的执行模型、跨核数据传递通路、任务错位流水。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 执行模型：一个逻辑 Block = 1 AIC + 2 AIV

Kernel 同时包含 `section_cube()` 与 `section_vector()` 时即为**混合 Kernel**。一个逻辑 Block 对应 1 个 AIC 和 2 个 AIV：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">执行单元</th><th style="text-align:left;">主要计算</th><th style="text-align:left;">片上存储</th><th style="text-align:left;">任务索引</th></tr>
<tr><td>AIC</td><td>矩阵乘加</td><td>L1 / L0A / L0B / L0C</td><td>get_block_idx()</td></tr>
<tr><td>AIV0 / AIV1</td><td>逐元素、归约、重排、Reg 计算</td><td>UB / Vector Register</td><td>get_block_idx()（AIV 全局编号）；get_subblock_idx()（组内编号）</td></tr>
</table>

要点：

- `block_dim` 配置的是 **AIC/AIV 执行组数**；AIC 数量 = `get_block_num()`，AIV 数量为执行组数的 2 倍；`get_subblock_num()` 在 MIX 的 AIV 侧返回 2，在 AIC 侧返回 1；
- 两个 AIV **执行相同的 Vector 域代码**，用 `get_subblock_idx()`（0/1）区分各自负责的数据；
- 多执行组场景下，**Cube 侧与 Vector 侧必须使用一致的任务映射**（可用 `get_block_idx() // get_subblock_num()` 对齐：AIC 侧除以 1，AIV 侧除以 2），避免不同执行组读写同一中间结果。


## 2. 跨核数据传递：两条通路

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">通路</th><th style="text-align:left;">路径</th><th style="text-align:left;">特点</th></tr>
<tr><td><b>片上 Buffer</b></td><td>Cube→Vec：L0C →(move)→ UB；Vec→Cube：UB →(move/insert)→ L1</td><td>不落 GM，可减少 GM 流量；生产/消费端的 shape、dtype、layout 必须符合搬运及拆分模式；一个 AIC 向两个 AIV 分发时按 M 轴或 N 轴拆分（AccToVecMode）</td></tr>
<tr><td><b>GM workspace</b></td><td>Cube：L0C→workspace→UB：Vector（或反向）</td><td>灵活（分块顺序可不同、可重复读）；代价是 GM 读写；并发单元用不重叠区域，复用需等读完再写</td></tr>
</table>

选择依据：中间数据小、两端口径一致 → 片上通路；中间数据大、需要乱序或重复读取 → workspace。


## 3. 跨核同步（回顾与深化）

第 2 章的规则在融合场景全部适用，并新增两条：

1. **源码顺序不能替代核间同步**：`section_cube()` 写在 `section_vector()` 前面，不代表 Vector 会等 Cube；
2. **auto_mutex 只管核内**：`auto_mutex=True` 负责 Tile 相关的核内流水同步，本节手动流水的核间（AIC↔AIV）依赖需显式 `set/wait_cross_core`；第 8 章自动 CV pipeline 另有跨核事件生成机制。

```python
with pl.section_cube():
    pl.store(workspace, cube_result, offsets)
    pl.system.set_cross_core(pipe=pl.PipeType.FIX, event_id=0)

with pl.section_vector():
    pl.system.wait_cross_core(pipe=pl.PipeType.MTE2, event_id=0)
    pl.load(vector_input, workspace, offsets)
```

事件规则：**成对出现；各参与端在对应动态迭代中必须执行匹配的 set/wait，不能单边跳过**（否则死锁）；同一 ID 复用前，前一组 SET/WAIT 必须全部消费完成。


## 4. 流水编排：融合的性能引擎

融合的收益不止"省一次 launch"——让 Cube 和 Vector **在同一轮处理不同任务**，可减少彼此等待；实际重叠程度受阶段耗时、搬运和同步开销影响：

```text
流水轮次     0       1       2       3       4
Cube       C(0)    C(1)    C(2)    C(3)      -
Vector       -     V(0)    V(1)    V(2)    V(3)
```

同一任务仍按 C(0)→V(0) 依赖执行；不同任务重叠——第 1 轮里 Cube 算任务 1、Vector 处理任务 0。

**task_id 编号法**：找出第一阶段每次交给下一阶段的那份数据，每产生一份就连续编号：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">task_id</th><th style="text-align:left;">m_idx</th><th style="text-align:left;">n_idx</th><th style="text-align:left;">含义</th></tr>
<tr><td>0</td><td>0</td><td>0</td><td>第 0 个输出 Tile</td></tr>
<tr><td>1</td><td>0</td><td>1</td><td>第 1 个输出 Tile</td></tr>
<tr><td>2</td><td>0</td><td>2</td><td>第 2 个输出 Tile</td></tr>
<tr><td>3</td><td>1</td><td>0</td><td>第 3 个输出 Tile</td></tr>
</table>

判断编号位置的方法：**看哪个循环变量推进后会生成下一份需要跨核传递的数据**（上例中内层 n_idx 每推进一次生成一份）。


### 一个 128×128 任务交给两个 AIV

完整练习规格为 [256,128]×[128,256]。输出按 128×128 划为 4 个任务，task//2 决定行块，task%2 决定列块。每个任务由一个 AIC 计算；两个 AIV 分别处理局部行 [0,64)、[64,128)，因此 AIV 的全局输出行起点为 `(task//2)*128 + sub_id*64`。

例如任务 3 对应输出右下角：AIV0 写 [128:192,128:256]，AIV1 写 [192:256,128:256]。两个 AIV 读取同一列段的 bias，各自广播到 64 行。事件保证结果已到 workspace，行分工保证写入不重叠，两者都必不可少。


### 从分工到完整 Kernel

下面给出上述规格的单槽 GEMM+Bias。一次启动使用 `kernel[None,1]`，表示一个 MIX 逻辑 Block，两个 AIV 按 sub_id 分行。bias 在 Host 整理成 [1,256] 行视图，数值语义仍是每列一个偏置。配套 Host 和后续 GELU/双槽版本见本章实践的独立源码。


In [ ]:
@pl.jit(auto_mutex=True)
def gemm_bias(a: pl.Tensor[[256,128],pl.DT_FP16],
           b: pl.Tensor[[128,256],pl.DT_FP16],
           bias: pl.Tensor[[1,256],pl.DT_FP32],
           workspace: pl.Tensor[[1,128,128],pl.DT_FP32],
           out: pl.Tensor[[256,256],pl.DT_FP32]):
    a1 = pl.make_tile_group(type=pl.TileType(shape=[128,128],dtype=pl.DT_FP16,target_memory=pl.MemorySpace.Mat),addrs=0,mutex_ids=[0])
    b1 = pl.make_tile_group(type=pl.TileType(shape=[128,128],dtype=pl.DT_FP16,target_memory=pl.MemorySpace.Mat),addrs=0x8000,mutex_ids=[1])
    al = pl.make_tile_group(type=pl.TileType(shape=[128,128],dtype=pl.DT_FP16,target_memory=pl.MemorySpace.Left),addrs=0,mutex_ids=[2])
    br = pl.make_tile_group(type=pl.TileType(shape=[128,128],dtype=pl.DT_FP16,target_memory=pl.MemorySpace.Right),addrs=0,mutex_ids=[3])
    ac = pl.make_tile_group(type=pl.TileType(shape=[128,128],dtype=pl.DT_FP32,target_memory=pl.MemorySpace.Acc),addrs=0,mutex_ids=[4])
    vx = pl.make_tile_group(type=pl.TileType(shape=[64,128],dtype=pl.DT_FP32,target_memory=pl.MemorySpace.Vec),addrs=0,mutex_ids=[5])
    vy = pl.make_tile_group(type=pl.TileType(shape=[64,128],dtype=pl.DT_FP32,target_memory=pl.MemorySpace.Vec),addrs=0x8000,mutex_ids=[6])
    vb = pl.make_tile_group(type=pl.TileType(shape=[1,128],dtype=pl.DT_FP32,target_memory=pl.MemorySpace.Vec),addrs=0x10000,mutex_ids=[7])
    with pl.section_cube():
        for task in pl.range(0,4):
            slot = task % 1
            if task >= 1:
                pl.system.wait_cross_core(pipe=pl.PipeType.M,event_id=slot+2)
            row = (task // 2)*128
            col = (task % 2)*128
            ta = a1.current()
            tb = b1.current()
            left = al.current()
            right = br.current()
            result = ac.current()
            pl.load(ta,a,[row,0])
            pl.load(tb,b,[0,col])
            pl.move(left,ta)
            pl.move(right,tb)
            pl.matmul(result,left,right)
            pl.store(workspace,result,[slot,0,0])
            pl.system.set_cross_core(pipe=pl.PipeType.FIX,event_id=slot)
        # Consume final releases; no unmatched events remain at return.
        for slot in pl.range(0,1):
            pl.system.wait_cross_core(pipe=pl.PipeType.M,event_id=slot+2)
    with pl.section_vector():
        sub = pl.get_subblock_idx()
        for task in pl.range(0,4):
            slot = task % 1
            row = (task // 2)*128 + sub*64
            col = (task % 2)*128
            tx = vx.current()
            ty = vy.current()
            bias_tile = vb.current()
            pl.system.wait_cross_core(pipe=pl.PipeType.MTE2,event_id=slot)
            pl.load(tx,workspace,[slot,sub*64,0])
            pl.load(bias_tile,bias,[0,col])
            pl.neg(bias_tile,bias_tile)  # 仅修改 UB Tile，GM 中的 bias 不变
            pl.expand_sub(ty,tx,bias_tile,dim=1)  # tx - (-bias)
            # Bias-only result stays in ty.
            pl.store(out,ty,[row,col])
            # Conservative release: include postprocessing and output store.
            pl.system.set_cross_core(pipe=pl.PipeType.MTE3,event_id=slot+2)


## 5. 课后练习

1. （填空）一个逻辑 Block 内 AIC 与 AIV 的数量比是 ______；两个 AIV 靠 ______ 区分各自数据。
2. （选择）`section_cube()` 代码写在 `section_vector()` 之前，能否保证 Vector 等 Cube 写完？
   A. 能，源码顺序即执行顺序<br/>
   B. 不能，核间依赖必须显式 set/wait_cross_core
3. （实践）两阶段流水 C→V，任务总数 4，第 0 轮 Vector 空转。画出 5 个轮次的执行表，指出总耗时相对串行（8 拍）节省了多少（假设每任务每阶段 1 拍）。

In [ ]:
!cat ./answer/05.02_answer.txt

## 6. 本节小结

- 混合 Kernel：1 AIC + 2 AIV，`get_subblock_idx()` 分数据，两侧任务映射必须一致；
- 跨核传递两通路：**片上 move（快、口径须一致）vs GM workspace（灵活、多 GM 往返）**；
- 核间同步显式做（源码顺序与 auto_mutex 都不管核间）；事件在参与端的动态迭代中配对；
- **任务错位流水 + task_id 连续编号**是融合性能的引擎。